In [2]:
from circuit_opt import *
from benchmark import *

In [3]:
nq_list = [6, 8, 10]
depth_list = [20, 40]
cx_density_list = [0.1, 0.3, 0.5, 0.7, 0.9]

processors_for_eval = 2
remote_weight = 5.0
local_weight = 1.0

# ---- pipeline wrappers ----
def _transpile_wrapper(qc: QuantumCircuit) -> PipelineResult:
    return transpile_only_pipeline(qc, name="transpile_only")

def _zx_wrapper(qc: QuantumCircuit) -> PipelineResult:
    return lc_pipeline(qc, name="zx_full_reduce")

def _part_v1_wrapper(qc: QuantumCircuit) -> PipelineResult:
    return partitioned_lc_pipeline(qc, processors=processors_for_eval, name="partitioned_lc_v1")

def _part_v2_wrapper(qc: QuantumCircuit) -> PipelineResult:
    return partitioned_lc_pipeline_v2(
        qc,
        processors=processors_for_eval,
        w_remote=remote_weight,
        w_local=local_weight,
        name="partitioned_lc_v2",
    )

pipelines = [_transpile_wrapper, _zx_wrapper, _part_v1_wrapper, _part_v2_wrapper]

In [4]:
samples_random: list[BenchmarkSample] = []
for pipe_fn in pipelines:
    samples = benchmark_pipeline_on_random(
        pipeline_fn=pipe_fn,
        num_qubits_list=nq_list,
        depth_list=depth_list,
        cx_density_list=cx_density_list,
        shots_per_setting=3,
        seed_offset=0,
        processors_for_eval=processors_for_eval,
        remote_weight=remote_weight,
        local_weight=local_weight,
    )
    samples_random.extend(samples)

/Users/parton/KEIO-UCL/QCircOptForPartitioning/LC_Partitioning/circuit_opt.py:114: RuntimeWarning: divide by zero encountered in matmul
  phase = np.angle(np.trace(np.conjugate(U2).T @ U1))
/Users/parton/KEIO-UCL/QCircOptForPartitioning/LC_Partitioning/circuit_opt.py:114: RuntimeWarning: overflow encountered in matmul
  phase = np.angle(np.trace(np.conjugate(U2).T @ U1))
/Users/parton/KEIO-UCL/QCircOptForPartitioning/LC_Partitioning/circuit_opt.py:114: RuntimeWarning: invalid value encountered in matmul
  phase = np.angle(np.trace(np.conjugate(U2).T @ U1))


In [6]:
import numpy as np

def summarize_eff_cost(
    samples: List[BenchmarkSample],
    label: str = "",
) -> None:
    """
    各パイプラインごとに eff_cost_after / eff_cost_before の
    平均・std・min・max を表示する。
    """
    print("=" * 60)
    print(f"Eff-cost summary {('- ' + label) if label else ''}")
    print("=" * 60)

    by_pipe: Dict[str, List[float]] = {}

    for s in samples:
        if s.eff_cost_before is None or s.eff_cost_before == 0:
            continue
        ratio = s.eff_cost_after / s.eff_cost_before
        by_pipe.setdefault(s.pipeline, []).append(ratio)

    for name, ratios in sorted(by_pipe.items()):
        arr = np.array(ratios, dtype=float)
        print(
            f"{name:16s}  "
            f"mean={arr.mean():.3f}, "
            f"std={arr.std():.3f}, "
            f"min={arr.min():.3f}, "
            f"max={arr.max():.3f}, "
            f"n={len(arr)}"
        )

    if not by_pipe:
        print("no eff_cost data found (eff_cost_before is None/0 everywhere)")

In [ ]:
def run_clustered_benchmark_for_pipeline(
    pipeline_fn,
    seed_offset: int = 1000,
) -> list[BenchmarkSample]:
    res_list: list[BenchmarkSample] = []
    base_seed = seed_offset

    for nq in nq_list:
        for d in depth_list:
            for dens in dens_list:  # dens は generator 内では使わないがインタフェース合わせで回す
                for s in range(3):
                    seed = base_seed + s

                    # クラスタ構造ありランダム回路
                    qc_rand = random_clustered_cx_circuit(
                        num_qubits=nq,
                        depth=d,
                        processors=cluster_processors,
                        intra_density=intra_density,
                        inter_density=inter_density,
                        seed=seed,
                    )
                    qc_rand = decompose_to_basis(qc_rand)

                    res = pipeline_fn(qc_rand)

                    # -------- remote / cost 評価 --------
                    remote_before = None
                    remote_after = None
                    eff_before = None
                    eff_after = None

                    if processors_for_eval is not None and processors_for_eval > 1:
                        base_circ = res.qc_in
                        n_eval = base_circ.num_qubits
                        p_eval = min(processors_for_eval, n_eval)
                        if p_eval < 1:
                            p_eval = 1

                        base_size = n_eval // p_eval
                        rem_q = n_eval % p_eval
                        sizes = [
                            base_size + (1 if i < rem_q else 0)
                            for i in range(p_eval)
                        ]

                        G = build_interaction_graph(base_circ)
                        part_of = kway_partition(G, sizes)

                        remote_before, local_before = count_remote_twoq(base_circ, part_of)
                        remote_after, local_after = count_remote_twoq(res.qc_out, part_of)

                        eff_before = remote_weight * remote_before + local_weight * local_before
                        eff_after = remote_weight * remote_after + local_weight * local_after

                    # -------- BenchmarkSample 追加 --------
                    res_list.append(
                        BenchmarkSample(
                            pipeline=res.name,
                            num_qubits=nq,
                            depth=d,
                            cx_density=dens,
                            seed=seed,
                            twoq_before=res.twoq_before,
                            twoq_after=res.twoq_after,
                            depth_before=res.depth_before,
                            depth_after=res.depth_after,
                            runtime_sec=res.runtime_sec,
                            remote_before=remote_before,
                            remote_after=remote_after,
                            eff_cost_before=eff_before,
                            eff_cost_after=eff_after,
                        )
                    )

    return res_list

samples_clustered: list[BenchmarkSample] = []
for i, pipe_fn in enumerate(pipelines):
    samples_clustered.extend(
        run_clustered_benchmark_for_pipeline(
            pipeline_fn=pipe_fn,
            seed_offset=1000 + 100 * i,
        )
    )

In [8]:
samples_clustered: list[BenchmarkSample] = []
for pipe_fn in pipelines:
    samples_clustered.extend(
        run_clustered_benchmark_for_pipeline(
            pipeline_fn=pipe_fn,
            seed_offset=1000,  # あなたが使っている値で
        )
    )
summarize_eff_cost(samples_clustered, label="clustered circuits")
summarize_eff_cost_wins(
    samples_clustered,
    label="clustered circuits",
    base_pipeline="transpile_only",
    target_pipeline="partitioned_lc_v2",
)

NameError: name 'run_clustered_benchmark_for_pipeline' is not defined

In [7]:
summarize_eff_cost(samples_random, label="random circuits")
summarize_eff_cost(samples_clustered, label="clustered circuits")

Eff-cost summary - random circuits
partitioned_lc_v1  mean=0.989, std=0.036, min=0.706, max=1.000, n=90
partitioned_lc_v2  mean=1.058, std=0.167, min=1.000, max=1.963, n=90
transpile_only    mean=0.906, std=0.084, min=0.676, max=1.000, n=90
zx_full_reduce    mean=1.594, std=0.637, min=0.489, max=2.852, n=90


NameError: name 'samples_clustered' is not defined

In [ ]:
summarize_eff_cost_wins(
    samples_random,
    label="random circuits",
    base_pipeline="transpile_only",
    target_pipeline="partitioned_lc_v2",
)